Day 11 – Precision Agriculture Using Spatial-Temporal Analytics on Databricks
Use Case: Crop Health Monitoring with Satellite NDVI + IoT Soil Moisture Data
🌾 Why Precision Agriculture Needs Spatial Intelligence

Traditional agriculture often treats entire fields as uniform blocks, but in reality, crop health varies spatially due to soil, irrigation, sunlight, pests, and other environmental factors. With modern sensors and satellite imagery, agri-businesses can now:

Detect zone-specific crop stress

Optimize irrigation and fertilization

Reduce water waste and prevent yield loss

Build predictive models for future health and harvest

This article shows how you can build an end-to-end pipeline in Databricks that brings together:

🌍 Satellite vegetation health (NDVI)

🌡 Soil moisture IoT devices

🌱 Field intelligence using grid-based spatial joins

🔁 Time-based crop monitoring with ML-ready output

All without requiring H3, external GIS tools, or separate pipelines.

🔁 Workflow Overview
(Simulated NDVI + Soil Data)
       ↓
(Bronze: Raw Data Load)
       ↓
(Silver: Spatial + Temporal Enrichment)
       ↓
(Gold: Crop Stress Detection by Grid)
       ↓
(Field Dashboard + Yield ML + Alerts)

✅ Step 1: Load Simulated NDVI + IoT Moisture Data (Python)

In [0]:
from pyspark.sql import Row
import random, uuid, time

# Simulate raw NDVI satellite readings
def generate_ndvi(n=200):
    base_lat, base_lon = 38.0, -121.0
    now = int(time.time())
    return [
        Row(
            ndvi_id=f"NDVI_{i}",
            lat=base_lat + random.random() * 0.1,
            lon=base_lon + random.random() * 0.1,
            ndvi_val=round(random.uniform(-0.1, 0.9), 2),
            ts=float(now - random.randint(0, 86400))
        )
        for i in range(n)
    ]

df_ndvi = spark.createDataFrame(generate_ndvi())
display(df_ndvi)


In [0]:
# Simulate soil moisture IoT readings
def generate_soil(n=200):
    base_lat, base_lon = 38.0, -121.0
    now = int(time.time())
    return [
        Row(
            sensor_id=f"SM_{i % 30}",
            lat=base_lat + random.random() * 0.1,
            lon=base_lon + random.random() * 0.1,
            moisture_pct=round(random.uniform(10, 90), 2),
            ts=float(now - random.randint(0, 86400))
        )
        for i in range(n)
    ]

df_soil = spark.createDataFrame(generate_soil())
display(df_soil)


These two datasets represent live satellite and field-level sensor data.

### ✅ Step 2: Add Spatial Enrichment & Timestamp Management

In [0]:
from pyspark.sql.functions import expr, concat, floor, col, lit

df_ndvi_silver = df_ndvi.withColumn(
    "event_time", expr("timestamp_millis(CAST(ts * 1000 AS BIGINT))")
).withColumn(
    "geo_point", expr("ST_POINT(lon, lat)")
).withColumn(
    "grid_id", concat(floor(col("lat") * 100), lit("_"), floor(col("lon") * 100))
)

display(df_ndvi_silver)



In [0]:
df_soil_silver = df_soil.withColumn(
    "event_time", expr("timestamp_millis(CAST(ts * 1000 AS BIGINT))")
).withColumn(
    "geo_point", expr("ST_POINT(lon, lat)")
).withColumn(
    "grid_id", concat(floor(col("lat") * 100), lit("_"), floor(col("lon") * 100))
)

display(df_soil_silver)


At this point, both datasets now have:

🧭 geo_point for spatial functions

🔲 grid_id for fast joins (1 km grid)

⏱️ event_time (human-readable)

### ✅ Step 3: Detect Crop Stress Zones (JOIN + AGGREGATE)

In [0]:
from pyspark.sql.functions import avg, when, current_date

# Join satellite NDVI and soil moisture data on spatial grid
joined = df_ndvi_silver.alias("n").join(
    df_soil_silver.alias("s"), 
    on="grid_id", 
    how="inner"
)

# Aggregate by grid and calculate crop condition score
crop_status = joined.groupBy("grid_id").agg(
    avg("ndvi_val").alias("avg_ndvi"),
    avg("moisture_pct").alias("avg_soil_moisture")
).withColumn(
    "crop_status",
    when((col("avg_ndvi") < 0.3) & (col("avg_soil_moisture") < 30), "SEVERE")
    .when((col("avg_ndvi") < 0.4) | (col("avg_soil_moisture") < 40), "MODERATE")
    .otherwise("HEALTHY")
).withColumn(
    "snapshot_date", current_date()
)

display(crop_status)


This produces a combined view of:

Vegetation health (NDVI)

Soil condition (moisture)

A computed crop status (Healthy, Moderate, Severe)

### Plot Crop Stress Zones (Folium Map)

In [0]:
%pip install folium

In [0]:
import folium

# Convert to Pandas and show on a map
gdf = crop_status.toPandas()
m = folium.Map(location=[38.05, -121.05], zoom_start=10)

for _, row in gdf.iterrows():
    color = "red" if row["crop_status"]=="SEVERE" else "orange" if row["crop_status"]=="MODERATE" else "green"
    lat = float(row["grid_id"].split("_")[0])/100
    lon = float(row["grid_id"].split("_")[1])/100
    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color=color,
        fill=True
    ).add_to(m)

m


Key Business Benefits
Value	Impact

- Prevent early-stage crop failure	Save 10–25% in seasonal losses
- Optimize irrigation via soil + NDVI patterns	Reduce water/energy costs
- Generate ML models to predict yield	Improve supply chain planning
- Focus farm agent visits on critical zones	Smarter labor allocation
- Build sustainability reports over time	Unlock eco-premium certifications